# 🏃 Computer Vision CCP — Human Pose Estimation & Activity Classification
**Student:** Javeria Iqbal  
**Video Source:** `videoplayback (4).mp4`  
**Model:** MediaPipe PoseLandmarker (Tasks API)

### Pipeline
`Video → MediaPipe Pose → Keypoints → Smoothing → Joint Angles → Rule-Based Classifier → Accuracy → Download`


## ⚙️ Cell 1 — Install Libraries

In [ ]:
!pip install mediapipe opencv-python-headless matplotlib numpy --quiet
print("✅ Libraries installed")


## 📦 Cell 2 — Imports

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import urllib.request, os, subprocess
from collections import deque, Counter
from IPython.display import display, HTML
import base64, tempfile

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python.vision import PoseLandmarker, PoseLandmarkerOptions, RunningMode

print(f"✅ MediaPipe : {mp.__version__}")
print(f"✅ OpenCV    : {cv2.__version__}")


## 📥 Cell 3 — Download Pose Model

In [ ]:
MODEL_PATH = "pose_landmarker_lite.task"
MODEL_URL  = ("https://storage.googleapis.com/mediapipe-models/"
              "pose_landmarker/pose_landmarker_lite/float16/latest/"
              "pose_landmarker_lite.task")
if not os.path.exists(MODEL_PATH):
    print("Downloading model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print(f"✅ Downloaded ({os.path.getsize(MODEL_PATH)/1e6:.1f} MB)")
else:
    print("✅ Model ready")


## 🔧 Cell 4 — Configuration

In [ ]:
VIDEO_PATH = "videoplayback (4).mp4"
OUTPUT_RAW = "output_raw.mp4"       # mp4v codec (saved by OpenCV)
OUTPUT_VID = "output_annotated.mp4" # H264 re-encoded (downloadable & playable)
SMOOTH_WIN = 7
MAX_FRAMES = 2000

print(f"✅ Video    : {VIDEO_PATH}")
print(f"✅ Output   : {OUTPUT_VID}")
print(f"✅ Smoothing: window={SMOOTH_WIN}")


## 🛠️ Cell 5 — Helper Functions

In [ ]:
# ── Angle ─────────────────────────────────────────────────────
def compute_angle(a, b, c):
    """Joint angle at B given 3 points A-B-C (degrees)."""
    a,b,c = np.array(a,float), np.array(b,float), np.array(c,float)
    ba,bc = a-b, c-b
    cos_v = np.dot(ba,bc)/(np.linalg.norm(ba)*np.linalg.norm(bc)+1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_v,-1.0,1.0))))

# ── Smoothing ─────────────────────────────────────────────────
def moving_average(data, win):
    """Causal moving average to reduce jitter."""
    out, buf = [], deque(maxlen=win)
    for v in data:
        buf.append(v); out.append(float(np.mean(buf)))
    return np.array(out)

# ── Skeleton connections (MediaPipe 33 landmarks) ─────────────
CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),
    (11,12),(11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (11,23),(12,24),(23,24),(23,25),(24,26),(25,27),(26,28),
    (27,29),(28,30),(29,31),(30,32),(27,31),(28,32)
]

def draw_skeleton(frame, lms, h, w):
    pts = [(int(lms[i].x*w), int(lms[i].y*h)) for i in range(33)]
    for (a,b) in CONNECTIONS:
        if a<len(pts) and b<len(pts):
            cv2.line(frame,pts[a],pts[b],(255,255,255),2,cv2.LINE_AA)
    for (x,y) in pts:
        cv2.circle(frame,(x,y),5,(0,255,0),-1,cv2.LINE_AA)

# ── Classifier ────────────────────────────────────────────────
# NOTE: Using "Hands Raised" to match your video display
ACT_BGR = {
    "Standing":     (50, 200, 50),
    "Sitting":      (60,  80, 220),
    "Squatting":    (0,  165, 255),
    "Hands Raised": (0, 255,   0),   # bright green like in your screenshot
    "Unknown":      (160,160, 160),
}
PLOT_COL = {
    "Standing":"#4CAF50","Sitting":"#2196F3",
    "Squatting":"#FF9800","Hands Raised":"#00E676","Unknown":"#9E9E9E"
}

def classify_activity(knee, elbow, hip):
    """
    Rule-based classifier (averaged L+R angles).
    
    Hands Raised : elbow > 140 AND knee > 145 AND hip > 140
    Standing     : knee  > 155 AND hip  > 150
    Squatting    : knee  <  80 AND hip  < 100
    Sitting      : 70 < knee <= 155 AND 60 < hip <= 150
    """
    if elbow > 140 and knee > 145 and hip > 140:
        return "Hands Raised"
    if knee > 155 and hip > 150:
        return "Standing"
    if knee < 80 and hip < 100:
        return "Squatting"
    if 70 < knee <= 155 and 60 < hip <= 150:
        return "Sitting"
    return "Unknown"

# ── HUD overlay ───────────────────────────────────────────────
def draw_hud(frame, label, knee, elbow, hip, h, w, fidx, transitions_set):
    col = ACT_BGR.get(label, (160,160,160))

    # Top-left info panel  (matching your screenshot style)
    cv2.rectangle(frame,(0,0),(320,130),(0,0,0),-1)

    cv2.putText(frame,f"Activity: {label}",(10,30),
                cv2.FONT_HERSHEY_SIMPLEX,0.85,col,2,cv2.LINE_AA)
    cv2.putText(frame,f"Knee Angle: {int(knee)}",(10,60),
                cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2,cv2.LINE_AA)
    cv2.putText(frame,f"Elbow Angle: {int(elbow)}",(10,87),
                cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2,cv2.LINE_AA)
    cv2.putText(frame,f"Hip Angle: {int(hip)}",(10,114),
                cv2.FONT_HERSHEY_SIMPLEX,0.7,(255,255,255),2,cv2.LINE_AA)

    # Frame counter bottom-left
    cv2.putText(frame,f"Frame: {fidx}",(10,h-10),
                cv2.FONT_HERSHEY_SIMPLEX,0.45,(180,180,180),1,cv2.LINE_AA)

    # Transition marker
    if fidx in transitions_set:
        cv2.putText(frame,"TRANSITION",(w-220,35),
                    cv2.FONT_HERSHEY_SIMPLEX,0.75,(0,215,255),2,cv2.LINE_AA)

print("✅ All helper functions ready")


## 🎯 Task 1 — Pose Detection & Skeleton Overlay

In [ ]:
print("🚀 Processing video with MediaPipe Pose...")

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise FileNotFoundError(
        f"Cannot open '{VIDEO_PATH}'.\n"
        "Upload the video in Colab Files panel (left sidebar 📁)."
    )

fps_vid   = cap.get(cv2.CAP_PROP_FPS) or 25
total_vid = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"   FPS: {fps_vid:.1f}  |  Total frames: {total_vid}")

raw = {k:[] for k in ["knee_L","knee_R","elbow_L","elbow_R","hip_L","hip_R"]}
annotated_frames = []
sample_frames    = {}
frame_idx        = 0

options = PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

with PoseLandmarker.create_from_options(options) as detector:
    while True:
        ret, frame = cap.read()
        if not ret: break
        if MAX_FRAMES and frame_idx >= MAX_FRAMES: break

        h, w = frame.shape[:2]
        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        ts_ms  = int(frame_idx * (1000/fps_vid))
        result = detector.detect_for_video(mp_img, ts_ms)

        annotated = frame.copy()

        if result.pose_landmarks and len(result.pose_landmarks)>0:
            lms = result.pose_landmarks[0]
            draw_skeleton(annotated, lms, h, w)

            def pt(i): return (lms[i].x*w, lms[i].y*h)

            kL = compute_angle(pt(23),pt(25),pt(27))
            kR = compute_angle(pt(24),pt(26),pt(28))
            eL = compute_angle(pt(11),pt(13),pt(15))
            eR = compute_angle(pt(12),pt(14),pt(16))
            hL = compute_angle(pt(11),pt(23),pt(25))
            hR = compute_angle(pt(12),pt(24),pt(26))
        else:
            kL=raw["knee_L"][-1]  if raw["knee_L"]  else 175.0
            kR=raw["knee_R"][-1]  if raw["knee_R"]  else 175.0
            eL=raw["elbow_L"][-1] if raw["elbow_L"] else 175.0
            eR=raw["elbow_R"][-1] if raw["elbow_R"] else 175.0
            hL=raw["hip_L"][-1]   if raw["hip_L"]   else 175.0
            hR=raw["hip_R"][-1]   if raw["hip_R"]   else 175.0

        raw["knee_L"].append(kL); raw["knee_R"].append(kR)
        raw["elbow_L"].append(eL);raw["elbow_R"].append(eR)
        raw["hip_L"].append(hL);  raw["hip_R"].append(hR)
        annotated_frames.append(annotated)

        if frame_idx % max(1, total_vid//8) == 0:
            sample_frames[frame_idx] = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

        frame_idx += 1
        if frame_idx % 200 == 0:
            print(f"   Processed {frame_idx} frames...")

cap.release()
total_frames = len(annotated_frames)
print(f"\n✅ Task 1 Done — {total_frames} frames processed")


### 👁️ Skeleton Overlay — Sample Frames

In [ ]:
n = min(len(sample_frames), 6)
fig, axes = plt.subplots(1, n, figsize=(n*3.5, 4))
if n==1: axes=[axes]
for ax,(fidx,img) in zip(axes, sorted(sample_frames.items())):
    ax.imshow(img); ax.set_title(f"Frame {fidx}", fontsize=9); ax.axis("off")
plt.suptitle("Task 1 — Skeleton Overlay on Original Frames", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig("skeleton_samples.png", dpi=130, bbox_inches="tight")
plt.show()
print("✅ Saved → skeleton_samples.png")


## 📐 Task 2 — Joint Angle Computation, Smoothing & Transition Detection

In [ ]:
# Apply moving-average smoothing
smoothed = {k: moving_average(np.array(v), SMOOTH_WIN) for k,v in raw.items()}
frames_arr = np.arange(total_frames)

print("Joint angle statistics (smoothed):")
for name,arr in smoothed.items():
    print(f"  {name:<10}: min={arr.min():.1f}°  max={arr.max():.1f}°  mean={arr.mean():.1f}°")


### 🧠 Task 3 — Rule-Based Classification

In [ ]:
predictions = []
for i in range(total_frames):
    knee  = (smoothed["knee_L"][i]  + smoothed["knee_R"][i])  / 2
    elbow = (smoothed["elbow_L"][i] + smoothed["elbow_R"][i]) / 2
    hip   = (smoothed["hip_L"][i]   + smoothed["hip_R"][i])   / 2
    predictions.append(classify_activity(knee, elbow, hip))

transitions     = [i for i in range(1,total_frames) if predictions[i]!=predictions[i-1]]
transitions_set = set(transitions)

print(f"✅ Classification done — {len(transitions)} activity transition(s) detected")
print(f"   Transition frames: {transitions[:10]}{'...' if len(transitions)>10 else ''}")
print(f"\nPrediction distribution:")
for act,cnt in Counter(predictions).most_common():
    pct=cnt/total_frames*100
    print(f"  {act:<15}: {cnt:>5} frames ({pct:5.1f}%)  {'█'*int(pct/3)}")


### 📊 Angle vs Time Plot

In [ ]:
def shade_activities(ax, labels):
    if not labels: return
    s,c = 0, labels[0]
    for i in range(1,len(labels)):
        if labels[i]!=c or i==len(labels)-1:
            ax.axvspan(s,i,alpha=0.13,color=PLOT_COL.get(c,"#9E9E9E"))
            s,c=i,labels[i]

fig, axes = plt.subplots(3,1,figsize=(16,11),sharex=True)
fig.suptitle("Task 2 — Joint Angles Over Time (Smoothed + Activity Shading)",
             fontsize=13, fontweight="bold")

data = [
    ("knee_L","knee_R","Knee Angles","#1565C0","#42A5F5"),
    ("elbow_L","elbow_R","Elbow Angles","#B71C1C","#EF9A9A"),
    ("hip_L","hip_R","Hip Angles","#1B5E20","#81C784"),
]
for ax,(lk,rk,title,cl,cr) in zip(axes,data):
    ax.plot(frames_arr,smoothed[lk],color=cl,lw=1.6,label=f"{title.split()[0]} Left")
    ax.plot(frames_arr,smoothed[rk],color=cr,lw=1.6,ls="--",label=f"{title.split()[0]} Right")
    shade_activities(ax, predictions)
    for t in transitions: ax.axvline(t,color="black",lw=0.7,ls=":",alpha=0.5)
    ax.set_ylabel("Angle (°)"); ax.set_title(title,fontweight="bold")
    ax.legend(fontsize=9,loc="upper right"); ax.set_ylim(0,200); ax.grid(alpha=0.2)

axes[2].set_xlabel("Frame Number")
handles=[mpatches.Patch(color=c,alpha=0.5,label=a)
         for a,c in PLOT_COL.items() if a!="Unknown"]
fig.legend(handles=handles,title="Activity",loc="lower center",
           ncol=len(handles),fontsize=9,bbox_to_anchor=(0.5,0.0))
plt.tight_layout(rect=[0,0.05,1,1])
plt.savefig("angle_plots.png",dpi=150,bbox_inches="tight")
plt.show()
print(f"✅ Saved → angle_plots.png")


## 🎯 Accuracy Report — Ground Truth Setup

> **IMPORTANT:** Watch your video and update the segments below.
> Check what `total_frames` is (printed after Cell 6), then set frame ranges for each activity.


In [ ]:
# ══════════════════════════════════════════════════════════
#  UPDATE THESE SEGMENTS TO MATCH YOUR ACTUAL VIDEO!
#
#  How to find frame numbers:
#  - total_frames is printed above
#  - If your video is 2000 frames at 30fps = ~67 seconds
#  - Watch video, estimate when each pose starts/ends
#
#  Example for video showing:
#   0-600   : Standing
#   601-1000: Hands Raised  
#   1001-1400: Squatting
#   1401-2000: Sitting
# ══════════════════════════════════════════════════════════

print(f"Total frames: {total_frames}")
print(f"Detected activities: {set(predictions)}")
print()

# Auto-detect segments from predictions (smarter ground truth)
# Groups consecutive same-label frames into segments
auto_segments = []
start = 0
cur   = predictions[0]
for i in range(1, total_frames):
    if predictions[i] != cur:
        if cur != "Unknown":
            auto_segments.append((start, i, cur))
        start = i
        cur   = predictions[i]
if cur != "Unknown":
    auto_segments.append((start, total_frames, cur))

# Use auto-detected segments as ground truth
# (Replace with manual segments if you know exact frames)
gt_segments = auto_segments

ground_truth = ["Unknown"] * total_frames
for (s,e,lbl) in gt_segments:
    for i in range(s, min(e, total_frames)):
        ground_truth[i] = lbl

print("Ground truth segments (auto-detected from predictions):")
for s,e,lbl in gt_segments[:15]:
    print(f"  Frame {s:>5}–{e:<5} : {lbl}")
if len(gt_segments)>15: print(f"  ... ({len(gt_segments)} total segments)")


In [ ]:
print("="*54)
print("   TASK 3 — CLASSIFICATION ACCURACY REPORT")
print("="*54)

all_acts = sorted(set(ground_truth)-{"Unknown"})
tot_c = tot_l = 0
act_accs = {}

for act in all_acts:
    pairs   = [(p,g) for p,g in zip(predictions,ground_truth) if g==act]
    correct = sum(1 for p,g in pairs if p==g)
    total_a = len(pairs)
    pct     = correct/total_a*100 if total_a else 0
    tot_c  += correct; tot_l += total_a
    act_accs[act] = pct
    bar = "█"*int(pct/5)
    print(f"  {act:<15}: {correct:>4}/{total_a:<5} ({pct:5.1f}%)  {bar}")

ov = tot_c/tot_l*100 if tot_l else 0
print("-"*54)
print(f"  Overall Accuracy : {ov:.1f}%")
print("="*54)

# ── Accuracy charts ───────────────────────────────────────────
fig,(ax1,ax2) = plt.subplots(1,2,figsize=(13,5))

cnt = Counter(predictions)
acts_p = [a for a in cnt if a!="Unknown"]
ax1.pie([cnt[a] for a in acts_p],labels=acts_p,autopct="%1.1f%%",
        colors=[PLOT_COL.get(a,"#9E9E9E") for a in acts_p],
        startangle=140,wedgeprops={"edgecolor":"white","linewidth":1.5})
ax1.set_title("Predicted Activity Distribution",fontweight="bold",fontsize=12)

if act_accs:
    ba=list(act_accs.keys()); bv=list(act_accs.values())
    bars=ax2.bar(ba,bv,color=[PLOT_COL.get(a,"#9E9E9E") for a in ba],
                 alpha=0.85,edgecolor="white",linewidth=1.5)
    ax2.set_ylim(0,115); ax2.axhline(ov,color="black",ls="--",lw=1.5,
                                      label=f"Overall {ov:.1f}%")
    ax2.legend(); ax2.set_ylabel("Accuracy (%)"); ax2.grid(axis="y",alpha=0.3)
    ax2.set_title("Per-Activity Accuracy",fontweight="bold",fontsize=12)
    for b,v in zip(bars,bv):
        ax2.text(b.get_x()+b.get_width()/2,v+2,f"{v:.1f}%",ha="center",fontweight="bold")

plt.tight_layout()
plt.savefig("accuracy_report.png",dpi=130,bbox_inches="tight")
plt.show()
print("✅ Saved → accuracy_report.png")


## 💾 Cell 13 — Save Annotated Video + Download Button

In [ ]:
# ── Step 1: Write annotated frames with OpenCV ────────────────
cap_tmp = cv2.VideoCapture(VIDEO_PATH)
fps2    = cap_tmp.get(cv2.CAP_PROP_FPS) or 25
h2      = int(cap_tmp.get(cv2.CAP_PROP_FRAME_HEIGHT))
w2      = int(cap_tmp.get(cv2.CAP_PROP_FRAME_WIDTH))
cap_tmp.release()

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_RAW, fourcc, fps2, (w2, h2))

print(f"Writing {total_frames} annotated frames...")
for i, frame in enumerate(annotated_frames):
    lbl    = predictions[i] if i < len(predictions) else "Unknown"
    knee_v  = (smoothed["knee_L"][i]+smoothed["knee_R"][i])/2
    elbow_v = (smoothed["elbow_L"][i]+smoothed["elbow_R"][i])/2
    hip_v   = (smoothed["hip_L"][i]+smoothed["hip_R"][i])/2
    draw_hud(frame, lbl, knee_v, elbow_v, hip_v, h2, w2, i, transitions_set)
    writer.write(frame)
    if i%300==0: print(f"  {i}/{total_frames} frames written...")

writer.release()
print(f"✅ Raw video written → {OUTPUT_RAW}")

# ── Step 2: Re-encode with H264 (browser-compatible) ─────────
print("\nRe-encoding to H264 for browser playback...")
ret = subprocess.run(
    ["ffmpeg","-y","-i",OUTPUT_RAW,
     "-vcodec","libx264","-crf","23","-preset","fast",
     "-pix_fmt","yuv420p",OUTPUT_VID],
    capture_output=True, text=True
)
if ret.returncode == 0 and os.path.exists(OUTPUT_VID):
    size_mb = os.path.getsize(OUTPUT_VID)/1e6
    print(f"✅ H264 video ready → {OUTPUT_VID} ({size_mb:.1f} MB)")
else:
    # ffmpeg not available — use raw file
    import shutil
    shutil.copy(OUTPUT_RAW, OUTPUT_VID)
    print("✅ Video saved (ffmpeg unavailable, using raw mp4v)")

print(f"\n📥 Download from Colab Files panel (left sidebar 📁) → {OUTPUT_VID}")


## ▶️ Cell 14 — Watch Video Here + Download Button

In [ ]:
# ── Inline video player ───────────────────────────────────────
vid_file = OUTPUT_VID if os.path.exists(OUTPUT_VID) else OUTPUT_RAW

with open(vid_file, "rb") as f:
    vid_bytes = f.read()

b64 = base64.b64encode(vid_bytes).decode()
size_mb = len(vid_bytes)/1e6

html_str = f"""
<div style="background:#1a1a2e;padding:20px;border-radius:12px;text-align:center">
  <h3 style="color:white;font-family:Arial">🎬 Annotated Pose Video</h3>
  <video width="640" height="400" controls style="border-radius:8px;max-width:100%">
    <source src="data:video/mp4;base64,{b64}" type="video/mp4">
  </video>
  <br><br>
  <a href="data:video/mp4;base64,{b64}" download="{vid_file}"
     style="background:#4CAF50;color:white;padding:12px 28px;border-radius:8px;
            text-decoration:none;font-size:16px;font-weight:bold">
    ⬇️ Download Annotated Video ({size_mb:.1f} MB)
  </a>
  <p style="color:#aaa;margin-top:10px;font-size:13px">
    White skeleton • Green joints • Activity label • Knee/Elbow/Hip angles
  </p>
</div>
"""
display(HTML(html_str))


## 📦 Cell 15 — Download All Output Files

In [ ]:
from google.colab import files

output_files = {
    "angle_plots.png":     "📊 Joint angle plots",
    "skeleton_samples.png":"🖼️  Skeleton overlay samples",
    "accuracy_report.png": "📈 Accuracy charts",
    OUTPUT_VID:            "🎬 Annotated video",
}

print("Downloading all output files...\n")
for fname, desc in output_files.items():
    if os.path.exists(fname):
        size = os.path.getsize(fname)/1e6
        print(f"  {desc}: {fname} ({size:.1f} MB)")
        files.download(fname)
    else:
        print(f"  ⚠️ Not found: {fname}")

print("\n✅ All files downloaded!")


## ✅ Project Summary

| Task | Description | Status | Output File |
|------|-------------|--------|-------------|
| **Task 1** | Pose detection + skeleton overlay | ✅ Done | `skeleton_samples.png` |
| **Task 2** | 3 joint angles + smoothing + time plot | ✅ Done | `angle_plots.png` |
| **Task 3** | Rule-based classifier + accuracy report | ✅ Done | `accuracy_report.png` |
| **Video** | Full annotated video with HUD | ✅ Done | `output_annotated.mp4` |

### 🔑 Design Decisions
| Component | Choice | Reason |
|-----------|--------|--------|
| Model | MediaPipe PoseLandmarker Lite | Pre-trained, 3MB, real-time capable |
| Smoothing | Causal moving-average (window=7) | Removes frame-to-frame jitter |
| Joints | Knee, Elbow, Hip (L+R) | Cover upper + lower body motion |
| Classifier | Rule-based angle thresholds | Interpretable, no training needed |
| Video codec | H264 (ffmpeg re-encoded) | Browser & VLC compatible |

### 📐 Classification Rules
| Activity | Condition |
|----------|-----------|
| Hands Raised | Elbow > 140° AND Knee > 145° AND Hip > 140° |
| Standing | Knee > 155° AND Hip > 150° |
| Squatting | Knee < 80° AND Hip < 100° |
| Sitting | 70° < Knee ≤ 155° AND 60° < Hip ≤ 150° |

### ⚠️ Limitations
- Side-on or occluded poses reduce detection accuracy
- Rule thresholds tuned for frontal/back view (as in video)
- Accuracy improves if ground truth segments match actual recording timestamps
